# 語料處理

        tokenizer.padding_side is left
        最後面有一個eos_token符號</s>
        注意:文章太長被截斷時，最後一個編碼沒有eos_token

        https://github.com/A-baoYang/alpaca-7b-chinese

 # Tokenizer

In [1]:
import torch
import transformers
from transformers import BloomTokenizerFast, AutoTokenizer
import datasets

In [2]:
tokenizer = BloomTokenizerFast.from_pretrained('Langboat/bloom-389m-zh')
#tokenizer = AutoTokenizer.from_pretrained('YeungNLP/bloomz-396m-zh', use_fast=True)
#tokenizer = BloomTokenizerFast.from_pretrained('Langboat/bloom-389m-zh', add_prefix_space=True)

# eos_token與pad_token如何設計?
    應該與Bloom原始設定一樣
    輸入文字編碼<unk><unk><unk> 後面都是padding符號(也是用0 <unk>)

    tokenizer.unk_token_id -> 0
    tokenizer.pad_token_id -> 3
    tokenizer.eos_token_id -> 2

In [3]:
tokenizer.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '</s>',
 'unk_token': '<unk>',
 'pad_token': '<pad>'}

# Prompt

def generate_prompt(data_point):

    # 這裡可以做繁簡轉換
    instruction = data_point["instruction"]
    input_text = data_point["input"]
    output_text = data_point["output"]
    result_prompt = f"Human: \n{instruction} {input_text}\nAssistant: \n\n{output_text}"
    return result_prompt

In [4]:
def generate_prompt(data_point):

    # 這裡可以做繁簡轉換
    instruction = data_point["instruction"]
    input_text = data_point["input"]
    output_text = data_point["output"]
    result_prompt = f"Human: \n{instruction} {input_text}\n\nAssistant: \n{output_text}"
    return result_prompt

In [5]:
data_point = {
    "instruction": "法国的首都是什么？",
    "input": "xx",
    "output": "法国的首都是巴黎。"
  }

In [6]:
print(generate_prompt(data_point))

Human: 
法国的首都是什么？ xx

Assistant: 
法国的首都是巴黎。


In [7]:
data_point = {
    "instruction": "",
    "input": "",
    "output": ""
  }

In [8]:
generate_prompt(data_point)

'Human: \n \n\nAssistant: \n'

# Tokenize

      每個樣本長度不一，因此編碼後的長度不一，若有超出最大長度者會被截斷。
      此處編碼不進行padding,padding工作交給後續的data_collator，進行padding即可(靠左)，每批次的長度以該批長度最長者為該批之整體句子長度。好處是:訓練時有些批之句子長度較短，較有效率。
      labels: 會在collator做批量時同時做padding(靠左)，labels之<pad>位置會被塞入"-100"
      attention_mask的值皆為1，若為<pad>則是0

In [9]:
# 編碼在最後一個加上一個\s eos_token符號  最前面並沒有bos_token
# 需要這樣做嗎?
# tokenizer.pad_token_id = 0  # unk. we want this to be different from the eos token

def tokenize(tokenizer, prompt, cutoff_len, add_eos_token=True):
    # there's probably a way to do this with the tokenizer settings
    # but again, gotta move fast
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=cutoff_len,
        padding=False,
        return_tensors=None,
    )
    if (
        result["input_ids"][-1] != tokenizer.eos_token_id
        and len(result["input_ids"]) < cutoff_len
        and add_eos_token
    ):
        result["input_ids"].append(tokenizer.eos_token_id)
        result["attention_mask"].append(1)

    # result["labels"] = copy.deepcopy(result["input_ids"])
    result["labels"] = result["input_ids"].copy()

    return result

cutoff_len = 300
def generate_and_tokenize_prompt(data_point):
    full_prompt = generate_prompt(data_point)
    tokenized_full_prompt = tokenize(tokenizer, full_prompt, cutoff_len) # 注意最後有一個</s>
    
    user_prompt = generate_prompt({**data_point, "output": ""})
    tokenized_user_prompt = tokenize(tokenizer, user_prompt, cutoff_len, add_eos_token=False)
    user_prompt_len = len(tokenized_user_prompt["input_ids"])

    tokenized_full_prompt["labels"] = [-100] * user_prompt_len + tokenized_full_prompt["labels"][user_prompt_len:]
    
    return tokenized_full_prompt

# def generate_and_tokenize_prompt(prompter, tokenizer, cutoff_len):
#     def generate_and_tokenize_prompt(data_point):
#         full_prompt = prompter.generate_prompt(
#             data_point["instruction"],
#             data_point["input"],
#             data_point["output"],
#         )
#         tokenized_full_prompt = tokenize(full_prompt, tokenizer, cutoff_len)
#         user_prompt = prompter.generate_prompt(data_point["instruction"], data_point["input"] )
#         tokenized_user_prompt = tokenize(user_prompt, tokenizer, cutoff_len, add_eos_token=False)
#         user_prompt_len = len(tokenized_user_prompt["input_ids"])
#         tokenized_full_prompt["labels"] = [-100] * user_prompt_len + tokenized_full_prompt["labels"][user_prompt_len:]  # could be sped up, probably
#         return tokenized_full_prompt
#     return generate_and_tokenize_prompt

# data collator

    bloom是靠左填充!! 這不一樣!!

    填充字元在左側，文字在右側。

    參數padding表示填充方式，可以為布爾類型、字符串類型或者一個PaddingStrategy對象。
    當值為布爾類型時，True表示填充至最大序列長度，False表示不填充。
    當為字符串類型時，"longest"表示填充值最大序列長度，"max_length"表示填充值參數max_length設置的長度，"do_not_pad"表示不填充。  
    
    參數max_length表示填充序列的最大長度，當設置padding="max_length"時，該參數才會有用。  
    
    參數pad_to_multiple_of表示填充的序列的倍數。  

    參數label_pad_token_id表示填充標籤時的值，默認為-100。注意，默認數據中序列填充的值為0，這與標籤填充的值不一致。

    label_pad_token_id (int, *optional*, defaults to -100):

    data collator會進行批量化，每批會以最長文句的長度，不足長度的樣本會填充字元(pad_token_id)在左側。

In [10]:
#  label_pad_token_id (int, *optional*, defaults to -100):
data_collator = transformers.DataCollatorForSeq2Seq(tokenizer, return_tensors="pt", padding=True)

In [11]:
data_collator

DataCollatorForSeq2Seq(tokenizer=BloomTokenizerFast(name_or_path='Langboat/bloom-389m-zh', vocab_size=42437, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False), model=None, padding=True, max_length=None, pad_to_multiple_of=None, label_pad_token_id=-100, return_tensors='pt')

# Read, preprocess, and save dataset

CUTOFF_LEN = 300 #256+41
dataset = datasets.load_dataset("json", data_files="C:\\Users\\a0936\\Downloads\\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\\alpaca_gpt4_data_zh_copy-簡體.json")
train_val_data = dataset["train"].train_test_split(test_size=0.015, shuffle=True, seed=42)


# 
train_data = train_val_data["train"].shuffle().map(generate_and_tokenize_prompt).remove_columns(['instruction', 'input', 'output'])
val_data = train_val_data["test"].shuffle().map(generate_and_tokenize_prompt).remove_columns(['instruction', 'input', 'output'])

# train_data = train_val_data["train"].shuffle().map(generate_and_tokenize_prompt_mask_labels).remove_columns(['instruction', 'input', 'output'])
# val_data = train_val_data["test"].shuffle().map(generate_and_tokenize_prompt_mask_labels).remove_columns(['instruction', 'input', 'output'])

train_data.save_to_disk('./train_dataset_Langboat簡體/data_train')
val_data.save_to_disk('./train_dataset_Langboat簡體/data_val')

CUTOFF_LEN = 300 #256+41
dataset = datasets.load_dataset("json", data_files="C:\\Users\\a0936\\Downloads\\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\\alpaca_gpt4_data_zh_copy-繁體.json")
train_val_data = dataset["train"].train_test_split(test_size=0.015, shuffle=True, seed=42)


# 
train_data = train_val_data["train"].shuffle().map(generate_and_tokenize_prompt).remove_columns(['instruction', 'input', 'output'])
val_data = train_val_data["test"].shuffle().map(generate_and_tokenize_prompt).remove_columns(['instruction', 'input', 'output'])

# train_data = train_val_data["train"].shuffle().map(generate_and_tokenize_prompt_mask_labels).remove_columns(['instruction', 'input', 'output'])
# val_data = train_val_data["test"].shuffle().map(generate_and_tokenize_prompt_mask_labels).remove_columns(['instruction', 'input', 'output'])

train_data.save_to_disk('./train_dataset_Langboat繁體/data_train')
val_data.save_to_disk('./train_dataset_Langboat繁體/data_val')

CUTOFF_LEN = 300 #256+41
dataset = datasets.load_dataset("json", data_files="C:\\Users\\a0936\\Downloads\\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\\alpaca_gpt4_data_zh_copy-簡體.json")
train_val_data = dataset["train"].train_test_split(test_size=0.015, shuffle=True, seed=42)


# 
train_data = train_val_data["train"].shuffle().map(generate_and_tokenize_prompt).remove_columns(['instruction', 'input', 'output'])
val_data = train_val_data["test"].shuffle().map(generate_and_tokenize_prompt).remove_columns(['instruction', 'input', 'output'])

# train_data = train_val_data["train"].shuffle().map(generate_and_tokenize_prompt_mask_labels).remove_columns(['instruction', 'input', 'output'])
# val_data = train_val_data["test"].shuffle().map(generate_and_tokenize_prompt_mask_labels).remove_columns(['instruction', 'input', 'output'])

train_data.save_to_disk('./train_dataset_YeungNLP簡體/data_train')
val_data.save_to_disk('./train_dataset_YeungNLP簡體/data_val')

In [12]:
CUTOFF_LEN = 300 #256+41
dataset = datasets.load_dataset("json", data_files="C:\\Users\\a0936\\Downloads\\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\\alpaca_gpt4_data_zh_copy-繁體.json")
train_val_data = dataset["train"].train_test_split(test_size=0.015, shuffle=True, seed=42)


# 
train_data = train_val_data["train"].shuffle().map(generate_and_tokenize_prompt).remove_columns(['instruction', 'input', 'output'])
val_data = train_val_data["test"].shuffle().map(generate_and_tokenize_prompt).remove_columns(['instruction', 'input', 'output'])

# train_data = train_val_data["train"].shuffle().map(generate_and_tokenize_prompt_mask_labels).remove_columns(['instruction', 'input', 'output'])
# val_data = train_val_data["test"].shuffle().map(generate_and_tokenize_prompt_mask_labels).remove_columns(['instruction', 'input', 'output'])

train_data.save_to_disk('./train_dataset_YeungNLP繁體/data_train')
val_data.save_to_disk('./train_dataset_YeungNLP繁體/data_val')

FileNotFoundError: Unable to find 'C:\Users\a0936\Downloads\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\w15-30-Alpaca-HumanAssistant短提示語(一問一答單輪對話)整理\alpaca_gpt4_data_zh_copy-繁體.json'